# PerfectPetClub - Data Engineering ETL Pipeline
## Customer Churn Prediction & Analytics

### 1. Project Overview & Business Problem
**PerfectPetClub** is an online subscription service providing pet food and supplies. A key business challenge is **customer churn**—when customers disable the auto-renewal feature of their subscription (`auto_renew = 0`).

This ETL pipeline extracts data across relational and transactional sources, validates data quality, engineers predictive features for churn analysis, and exports the final curated dataset into an optimized Apache Parquet format.

### 2. Architecture: Extract - Transform - Load (ETL)
* **Extract:** Ingest customer profiles, subscriptions, sales, payments, and product reviews from SQLite (`PerfectPetClub.db`).
* **Transform:**
  1. *Data Quality Validation:* Run automated assertion checks (null handling, date order logic, identifier uniqueness, boundary checks).
  2. *Feature Engineering:* Construct the binary churn target (`is_churned`), subscription tenure, financial aggregations, and customer feedback metrics.
  3. *Data Ethics & GDPR:* Mask/drop personally identifiable information (PII) such as passwords and street addresses.
* **Load:** Write the clean customer-level feature dataset to `output/perfectpetclub_churn_features.parquet`.

In [ ]:
import sys
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import logging

sys.path.append('../src')

from extract_customer_subscriptions import (
    extract_customer_subscriptions,
    validate_customer_data,
    transform_customer_features
)

print('Environment and modules loaded successfully!')


### Stage 1: Extract — Customer and Subscription Data

This stage extracts customer and subscription information from the SQLite database `PerfectPetClub.db`.

The extraction uses SQL INNER JOIN operations across three tables:

- **customer:** Customer ID, name, email, county and postcode.
- **subscription:** Subscription ID, start date, end date and auto-renewal status.
- **subscription_type:** Subscription tier and delivery service.

The extracted dataset contains 221 customer records, which are passed to the validation and feature engineering stages.


In [ ]:
DB_PATH = '../data/PerfectPetClub.db'

raw_customers = extract_customer_subscriptions(db_path=DB_PATH)

print(f'Extracted {len(raw_customers)} customer records.')
raw_customers.head(3)

## Stage 2: Transform — Automated Data Quality Checks
Before performing feature engineering, we validate data consistency, integrity, and domain rules:
1. **Uniqueness:** Verify all `customer_id` values are unique.
2. **Missing Values:** Ensure primary identifiers and renewal statuses have zero nulls.
3. **Date Logic:** Check that `sub_start_date` <= `sub_end_date`.
4. **Value Boundaries:** Check non-negative spend values and review ratings bounded within [0, 5].

In [ ]:
print("--- Validating Customer Data ---")

cust_dq = validate_customer_data(raw_customers)

print("Customer validation results:")
for check, result in cust_dq.items():
    print(f"{check}: {result}")

--- Validating Customer Data ---
  ✓ null_checks: {'customer_id': 0, 'email': 0, 'subscription_id': 0, 'auto_renew': 0}
  ✓ is_unique: True
  ✓ valid_auto_renew: True
  ✓ date_logic_valid: True

--- Validating Sales and Review Data ---
  ✓ valid_spend: True
  ✓ valid_orders: True
  ✓ valid_ratings: True


In [ ]:

# Test: Can our validation detect an invalid date?

test_data = raw_customers.copy()

# Deliberately insert an invalid subscription date
test_data.loc[0, 'sub_end_date'] = 'invalid-date'

try:
    validate_customer_data(test_data)
    print("TEST FAILED: Invalid date was accepted.")

except AssertionError as e:
    print("TEST PASSED: Invalid date was detected.")
    print(e)


### Stage 3: Transform — Customer Feature Engineering

In this stage, we create features from the customer and subscription data to support churn analysis.

- **is_churned:** Binary target variable (1 if auto-renew is disabled, otherwise 0).
- **subscription_tenure_days:** Number of days between the recorded subscription start and end dates.
- **postcode_area:** Leading letters of the postcode used for geographical analysis.

The transformed dataset is checked before integration with other team members' data.

In [ ]:
# 1. Transform your customer and subscription data
clean_customers = transform_customer_features(raw_customers)

# 2. Remove unnecessary identifiers
final_dataset = clean_customers.drop(
    columns=['customer_name', 'postcode', 'subscription_id']
)

# 3. Check your processed data
print("Total customers:", len(final_dataset))
print("Churned customers:", final_dataset['is_churned'].sum())
print("Dataset shape:", final_dataset.shape)

display(final_dataset.head())

# Churn rate by subscription tier

In [ ]:
# Analyse churn rate by subscription tier

tier_summary = final_dataset.groupby("subscription_tier")["is_churned"].agg(
    total_customers="count",
    churn_rate="mean"
).reset_index()

# Convert churn rate into percentage
tier_summary["churn_rate"] = (
    tier_summary["churn_rate"] * 100
).round(2)

# Display results
print("--- Churn Rate by Subscription Tier ---")
display(tier_summary)

## Stage 4: Load — Export Curated Dataset to Apache Parquet
As specified in Task 2, the final transformed dataset must be exported as an Apache Parquet file.
Parquet provides columnar compression, preserves column data types, and is optimized for downstream analytical queries.

In [ ]:
from pathlib import Path
import pandas as pd

# Create output folder
output_dir = Path("../output")
output_dir.mkdir(parents=True, exist_ok=True)

# Save only my customer and subscription features
parquet_path = output_dir / "customer_subscription_features.parquet"

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False)

# Read the saved file to verify
verified = pd.read_parquet(parquet_path)

print("Parquet saved successfully!")
print("Total customers:", len(verified))
print("Total columns:", len(verified.columns))
print("File location:", parquet_path.resolve())

# Verification: Read back parquet file to guarantee integrity

In [ ]:
# Verify the new customer-only Parquet file
df_verified = pd.read_parquet(parquet_path)

print("Parquet verification passed!")
print("Rows:", df_verified.shape[0])
print("Columns:", df_verified.shape[1])
print("Columns match:", list(df_verified.columns) == list(final_dataset.columns))

In [ ]:
# Final validation of customer subscription features

# 1. Check unique customer IDs
assert final_dataset["customer_id"].is_unique

# 2. Check customer count has not changed
assert len(final_dataset) == len(raw_customers)

# 3. Verify churn calculation
expected_churn = (
    final_dataset["auto_renew"] == 0
).astype(int)

assert (final_dataset["is_churned"] == expected_churn).all()

# 4. Check subscription durations
assert (final_dataset["subscription_tenure_days"] >= 0).all()

# 5. Verify exported Parquet structure
assert df_verified.shape == final_dataset.shape
assert list(df_verified.columns) == list(final_dataset.columns)

print("All 5 final validation checks PASSED!")